# Week 9, Day 1: Taylor series: approximating nonlinear functions

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w009/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Build Taylor polynomials

Write `taylor_eval(derivs, a, x)` where `derivs[k]` is the kth derivative of f at a. Use it for $e^x$ around 0 (every derivative is 1) and record the error at x = 1 for orders 0 to 7 in `errs`. Plot the errors on a log scale.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import factorial

def taylor_eval(derivs, a, x):
    return ...

errs = [abs(taylor_eval([1.0] * (n + 1), 0.0, 1.0) - np.e) for n in range(8)]
plt.figure(figsize=(5, 3)); plt.semilogy(range(8), errs, "o-"); plt.xlabel("order n"); plt.ylabel("error at x = 1"); plt.show()
print(np.array(errs).round(8))

In [ ]:
import numpy as np
def test_poly():
    "taylor_eval sums f^(k)(a) / k! (x - a)^k"
    assert np.isclose(taylor_eval([1.0, 2.0, 6.0], 1.0, 3.0), 1 + 2 * 2 + 3 * 4)

def test_exp():
    "the order-4 polynomial of exp around 0 is accurate to 1e-3 at x = 0.5"
    assert abs(taylor_eval([1.0] * 5, 0.0, 0.5) - np.exp(0.5)) < 1e-3

def test_errors():
    "errs[n] is the error of order n at x = 1, and it falls with n"
    assert len(errs) == 8 and np.all(np.diff(errs) < 0)

test_poly()
test_exp()
test_errors()
print("All checks passed")

### Exercise: Check the error constant

For $f = \sin$ at x = 1, compute the central-difference error divided by $h^2$ for each h in `hs` and store the values in `ratios`. They should settle at $f'''(1)/6 = -\cos(1)/6$.

In [ ]:
hs = [0.1, 0.03, 0.01, 0.003]
ratios = []
for h in hs:
    est = (np.sin(1 + h) - np.sin(1 - h)) / (2 * h)
    ratios.append(...)
print(np.round(ratios, 6), " predicted", round(-np.cos(1) / 6, 6))

In [ ]:
import numpy as np
def test_ratio():
    "ratios approach f'''(x) / 6 = -cos(1) / 6 for sin at x = 1"
    assert abs(ratios[-1] - (-np.cos(1.0) / 6)) < 1e-4

test_ratio()
print("All checks passed")

### Exercise: Taylor-expand the EIF neuron

Find `V_min`, where $F'(V) = -1 + \exp((V - V_T)/\Delta_T) = 0$. Compute `F0`, `F1` and `F2`, the values of F, F′ and F″ there, and write the quadratic approximation `Fq(V)`. Plot F and Fq. Over what range of voltages is the quadratic good?

In [ ]:
EL, V_T, dT = -65.0, -50.0, 2.0            # mV
def F(V):
    return -(V - EL) + dT * np.exp((V - V_T) / dT)

V_min = ...
F0 = F(V_min)
F1 = ...                                   # F'(V_min)
F2 = ...                                   # F''(V_min)
def Fq(V):
    return ...

V = np.linspace(-70, -44, 300)
plt.figure(figsize=(5, 3.2)); plt.plot(V, F(V), label="exponential"); plt.plot(V, Fq(V), "--", label="quadratic (Taylor)")
plt.axhline(0, color="0.6", lw=0.8); plt.ylim(-10, 25); plt.xlabel("V (mV)"); plt.ylabel("F(V) (mV)"); plt.legend(frameon=False); plt.show()
print(f"minimum of F at {V_min} mV, F = {F0:.2f}; curvature F'' = {F2:.2f}")

In [ ]:
import numpy as np
def test_point():
    "V_min is where F'(V) = 0, which is V_T"
    assert np.isclose(V_min, V_T)

def test_coeffs():
    "F0, F1 and F2 are F, F' and F'' at V_min"
    assert np.isclose(F0, F(V_T)) and np.isclose(F1, 0, atol=1e-12) and np.isclose(F2, 1 / dT)

def test_close():
    "the quadratic is within 0.5 mV of F within 2 mV of V_T"
    V = np.linspace(V_T - 2, V_T + 2, 41)
    assert np.max(np.abs(Fq(V) - F(V))) < 0.5

test_point()
test_coeffs()
test_close()
print("All checks passed")